In [ ]:
import os
import re
import shutil
import logging
import tempfile
import subprocess
from typing import Optional, Tuple
import openpyxl
from pypdf import PdfWriter
import ipywidgets as widgets
from IPython.display import display, HTML

# ----------------------------------------------------------------------
# CONFIGURACIÓN DE RUTAS PARA ENTORNO NUBE (MyBinder)
# ----------------------------------------------------------------------
# Al estar en Binder, la carpeta raíz "./" es donde están tus PDFs de calibración.
CAL_DIR = "./"

KNOWN_BRANDS = {
    'FLUKE', 'FLK', 'B2', 'OMICRON', 'MEGABRAS',
    'AEMC', 'KEYSIGHT', 'AGILENT', 'MEGGER',
    'EXTECH', 'UNI-T', 'GWINSTEK', 'RIGOL',
    'TEKTRONIX', 'HIOKI', 'KAISE',
    'CHAUVINARNOUX', 'CA', 'KYORITSU',
    'AMPROBE', 'IDEAL', 'GREENLEE',
    'MARTINDALE', 'SEAWARD'
}

# ----------------------------------------------------------------------
# FUNCIONES DE NORMALIZACIÓN Y BÚSQUEDA
# ----------------------------------------------------------------------
def normalize(s: str) -> str:
    if not s: return ""
    s = str(s).strip().upper()
    return re.sub(r'[\s\.\-_/\\:]+', '', s)

def normalize_date(date_str: str) -> str:
    if not date_str: return ""
    txt = str(date_str).strip()
    m = re.search(r'(\d{1,2})[\/\-](\d{1,2})[\/\-](\d{4})', txt)
    if not m: return ""
    dd = m.group(1).zfill(2)
    mm = m.group(2).zfill(2)
    yyyy = m.group(3)
    return f"{dd}{mm}{yyyy}"

def append_calibration_pdf(main_pdf_path: str, ws) -> str:
    if not os.path.exists(main_pdf_path) or not os.path.exists(CAL_DIR): 
        return main_pdf_path
    try:
        pdf_files = [f for f in os.listdir(CAL_DIR) if f.lower().endswith(".pdf")]
        pdf_info = []
        for fname in pdf_files:
            name_no_ext = os.path.splitext(fname)[0]
            pdf_info.append({
                "path": os.path.join(CAL_DIR, fname), 
                "name": name_no_ext, 
                "norm": normalize(name_no_ext)
            })
    except Exception as e:
        print(f"Error leyendo calibraciones: {e}")
        return main_pdf_path

    if not pdf_info: return main_pdf_path

    equipos = []
    current = {"brand": "", "model": "", "serial": "", "cal_date": ""}
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
        values = []
        for c in row:
            try: values.append(str(c.value).strip() if c.value else "")
            except: values.append("")
        row_text = " | ".join(values).upper()

        if "MARCA" in row_text or "BRAND" in row_text:
            for v in values:
                nv = normalize(v)
                for b in KNOWN_BRANDS:
                    if normalize(b) == nv: current["brand"] = normalize(b)
        if "MODELO" in row_text or "MODEL" in row_text:
            for v in values:
                nv = normalize(v)
                if re.search(r'[A-Z0-9]{3,}', nv) and 2 <= len(nv) <= 30:
                    if nv not in ("MODELO", "MODEL"): current["model"] = nv
        if "SERIE" in row_text or "SERIAL" in row_text:
            for v in values:
                nv = normalize(v)
                if len(nv) >= 5 and re.search(r'[A-Z0-9]{5,}', nv): current["serial"] = nv
        if "CALIBRACION" in row_text or "CALIBRATION" in row_text:
            for v in values:
                if not v: continue
                nd = normalize_date(v)
                if nd: current["cal_date"] = nd

        if sum([1 if current[f] else 0 for f in current]) >= 2:
            equipos.append(current.copy())
            current = {"brand": "", "model": "", "serial": "", "cal_date": ""}

    if not equipos: return main_pdf_path

    matched_paths = []
    for equipo in equipos:
        best_score, best_pdf = 0, None
        for pdf in pdf_info:
            score = 0
            if equipo["brand"] and equipo["brand"] in pdf["norm"]: score += 50
            if equipo["model"] and (equipo["model"] in pdf["norm"] or pdf["norm"] in equipo["model"]): score += 35
            if equipo["serial"] and equipo["serial"] in pdf["norm"]: score += 80
            if equipo["cal_date"] and equipo["cal_date"] in pdf["norm"]: score += 120
            if score > best_score: best_score, best_pdf = score, pdf
        if best_pdf and best_score >= 50 and best_pdf["path"] not in matched_paths:
            matched_paths.append(best_pdf["path"])

    if not matched_paths: return main_pdf_path

    try:
        writer = PdfWriter()
        with open(main_pdf_path, "rb") as f: writer.append(f)
        for pdf_path in matched_paths:
            with open(pdf_path, "rb") as f: writer.append(f)
        tmp_output = main_pdf_path.replace(".pdf", "_temp.pdf")
        with open(tmp_output, "wb") as out: writer.write(out)
        os.replace(tmp_output, main_pdf_path)
    except Exception as e:
        print(f"Error acoplando PDFs: {e}")
    return main_pdf_path

# ----------------------------------------------------------------------
# LÓGICA DE PROCESAMIENTO JUPYTER LOCAL
# ----------------------------------------------------------------------
def process_excel(excel_path: str, output_pdf_name: str):
    try:
        wb = openpyxl.load_workbook(excel_path, data_only=False)
        ws = wb.active
        
        # Mantiene tu configuración original de impresión exacta
        ws.print_area = '$A$1:$M$41'
        
        try:
            ws.page_setup.orientation = 'portrait'
            ws.page_setup.fitToWidth = 1
            ws.page_setup.fitToHeight = 0
            ws.page_setup.fitToPage = True
        except:
            pass
            
        try: ws.sheet_view.showGridLines = True
        except: pass
        
        wb.save(excel_path)
        
        # Conversión del motor local de Linux
        cmd = ["libreoffice", "--headless", "--convert-to", "pdf", excel_path]
        subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        
        generated_pdf = excel_path.replace(".xlsx", ".pdf")
        if os.path.exists(generated_pdf):
            final_pdf = append_calibration_pdf(generated_pdf, ws)
            shutil.move(final_pdf, output_pdf_name)
            wb.close()
            return True
        wb.close()
        return False
    except Exception as e:
        print(f"Error procesando el archivo: {e}")
        return False

# ----------------------------------------------------------------------
# INTERFAZ WEB GRÁFICA INTERACTIVA (Voila)
# ----------------------------------------------------------------------
uploader = widgets.FileUpload(
    accept='.xlsx',
    multiple=False,
    description='Subir Excel',
    button_style='primary'
)

output = widgets.Output()

def on_file_upload(change):
    with output:
        output.clear_output()
        if not uploader.value:
            return
            
        print("⏳ Procesando protocolo, por favor espere...")
        
        # Obtener archivo cargado por el usuario
        uploaded_file = uploader.value[0] if isinstance(uploader.value, list) else list(uploader.value.values())[0]
        file_name = uploaded_file['name']
        file_content = uploaded_file['content']
        
        base_name = os.path.splitext(file_name)[0]
        temp_input = f"temp_{file_name}"
        final_output_pdf = f"{base_name}.pdf"
        
        with open(temp_input, "wb") as f:
            f.write(file_content)
            
        success = process_excel(temp_input, final_output_pdf)
        
        if os.path.exists(temp_input):
            os.remove(temp_input)
            
        if success and os.path.exists(final_output_pdf):
            print(f"✅ ¡Éxito! El PDF '{final_output_pdf}' se generó correctamente.")
            # JavaScript para forzar la descarga en el navegador del usuario automáticamente
            js_download = f"""
            <script>
                var link = document.createElement('a');
                link.href = 'files/{final_output_pdf}';
                link.download = '{final_output_pdf}';
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);
            </script>
            """
            display(HTML(js_download))
        else:
            print("❌ Ocurrió un problema al intentar generar el archivo PDF.")

uploader.observe(on_file_upload, names='value')

# Renderizado de la app web limpia
display(HTML("<h2>Conversor Automático de Protocolos a PDF</h2>"))
display(HTML("<p>Subí tu archivo de Excel (.xlsx) y el sistema generará el PDF aplicando la escala exacta y adosando las calibraciones de manera automática.</p>"))
display(uploader, output)

2026-07-06 10:10:58,081 | INFO | Procesando: C:/Users/JoaquinAzpiroz/Downloads/LV temporal Cable ATS B1-2-b-21 (1).xlsx
2026-07-06 10:10:59,516 | INFO | Rango calculado → Filas 1-108 | Columnas 1-9
2026-07-06 10:10:59,518 | INFO | Área impresión: Filas 1-108, Cols 1-9
2026-07-06 10:11:14,238 | INFO | PDFs calibración encontrados: ['AEMC 6240 222976YLH 24102025', 'AEMC 6505 150165UKDV 18022026', 'AEMC 6550 111470YLDV 13052026', 'AEMC 6550 111470YLDV 22102025', 'AEMC 8510 DTR 113070ZADV 23102025', 'B2 HVA68-2 GH5228.22I001 01102025', 'B2 HVA68-2 GH5228.22I001 14052026', 'B2 PDTD60-2 GH5728.22A017 14052026', 'Baur Viola TD 1446418001 08062026', 'Eurotech 5530 - 10-120 Nm  250600157 03022026', 'Eurotech 5530 - 5-25 Nm 250600060 02022026', 'Eurotech 5530 - 60-330 Nm 250600292 02022026', 'Eurotech 5530 250600060 02022026', 'Eurotech 5530 250600157 03022026', 'Eurotech 5530 250600292 03022026', 'FLUKE 115 45160680SV 06042026', 'FLUKE 377 63700002 24102025', 'FLUKE T6-1000 59220035 13042026', 